# Vision v1: Fast VLM Execution Test
This notebook tests how fast a local VLM (like Qwen2-VL-2B) can process a screenshot and output click coordinates (schema).

In [ ]:
!pip install mlx-vlm Pillow pyautogui jinja2

In [ ]:
import time
import json
from PIL import ImageGrab
from mlx_vlm import load, generate
from mlx_vlm.prompt_utils import get_message_format
from mlx_vlm.utils import load_config

# 1. Load the model (Using 4-bit quantized Qwen2-VL-2B for max speed on Mac)
model_path = "Tzafon/Northstar-CUA-Fast"
print(f"Loading {model_path}...")
start_load = time.time()
model, processor = load(model_path, strict=False)
config = load_config(model_path)
print(f"Model loaded in {time.time() - start_load:.2f} seconds")

In [ ]:
# 2. Capture a quick screenshot
print("Capturing screenshot...")
img = ImageGrab.grab()
    img = img.convert("RGB")

# Optional: Downscale to save VRAM and inference time
img.thumbnail((1024, 1024)) 
img_path = "test_screenshot.jpg"
img.save(img_path)
print(f"Screenshot saved to {img_path}")

In [ ]:
# 3. Prepare the prompt
context = "You are a fast UI executor."
task = "Find the Apple menu icon (top left)."

prompt = f"{context} {task} Output ONLY a valid JSON object with keys \"x\" and \"y\" representing the coordinates to click, and \"status\" (done/failed). No other text."

messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
formatted_prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

# 4. Generate & Time it
print("Running inference...")
start_inf = time.time()

# Note: max_tokens is small since we only want a JSON schema
output = generate(model, processor, formatted_prompt, [img_path], verbose=False, max_tokens=50)

end_inf = time.time()

print("\n--- Results ---")
print(f"Inference Time: {end_inf - start_inf:.4f} seconds")
print(f"Raw Output:\n{output}")

try:
    # Try to parse it to ensure it is valid JSON
    parsed = json.loads(output.strip().strip("```json").strip("```"))
    print(f"\nParsed Schema: {parsed}")
except Exception as e:
    print(f"\nFailed to parse JSON: {e}")